In [26]:
import torch

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],
    ]  # step     (x^6)
)

In [27]:
x_2 = inputs[1]
# inputs ko second vector (A) select gareko

d_in = inputs.shape[1]
# input vector ko dimension (B) nikaleko

d_out = 2
# output vector ko size set gareko

In [28]:
torch.manual_seed(123)
# random numbers same aauna ko lagi seed set gareko

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# query weight matrix banako (trainable jasto structure)
# requires_grad=False le training ma update hudaina

W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# key weight matrix banako, attention compare garna use hunxa

W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# value weight matrix banako, final output represent garna use hunxa

In [29]:
query_2 = x_2 @ W_query
# x_2 lai query weight sanga multiply garera query vector banauxa

key_2 = x_2 @ W_key
# x_2 lai key weight sanga multiply garera key vector banauxa

value_2 = x_2 @ W_value
# x_2 lai value weight sanga multiply garera value vector banauxa
# actual information represent garxa

print(query_2)
# query vector print gareko

tensor([0.4306, 1.4551])


In [30]:
keys = inputs @ W_key
# inputs lai key weight sanga multiply garera sabai token ko key vector banauxa

values = inputs @ W_value
# inputs lai value weight sanga multiply garera sabai token ko value vector banauxa

print("keys.shape:", keys.shape)
# key matrix ko shape print gareko (kati token ra kati dimension cha bhanera dekhaunxa)

print("values.shape:", values.shape)
# value matrix ko shape print gareko (kati token ra kati dimension cha bhanera dekhaunxa)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [31]:
keys_2 = keys[1]
# keys ko second token select gareko

attn_score_22 = query_2.dot(keys_2)
# query ra key ko dot product garera attention score nikaleko
# yesle dherai similar xa vane high score dinxa

print(attn_score_22)
# final attention score print gareko

tensor(1.8524)


In [32]:
attn_scores_2 = query_2 @ keys.T
# query_2 lai sabai keys sanga multiply garera attention scores banauxa
# yesle x_2 le sabai tokens sanga kati relate garcha bhanera dekhaunxa

print(attn_scores_2)
# final attention score vector print gareko

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [33]:
d_k = keys.shape[-1]
# key vector ko dimension nikaleko (scale garna use hunxa)

attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
# attention scores lai scale garera softmax lagako
# sqrt(d_k) le values stable banauxa (too large hunu bata bachauxa)
# dim=-1 le last dimension (sab tokens) ma probability banauxa

print(attn_weights_2)
# final attention weights print gareko
# yesle kun token lai kati focus garne bhanera dekhaunxa

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [34]:
context_vec_2 = attn_weights_2 @ values
# attention weights lai values sanga multiply garera final context vector banauxa
# yesle x_2 le sabai tokens bata kati information lina parxa bhanera combine garxa

print(context_vec_2)
# final context vector print gareko
# yesle attention ko final output dekhaunxa

tensor([0.3061, 0.8210])


In [35]:
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    # self attention layer banako

    def __init__(self, d_in, d_out):
        super().__init__()
        # parent class initialize gareko

        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        # query weight matrix banako (trainable parameter)

        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        # key weight matrix banako

        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
        # value weight matrix banako

    def forward(self, x):
        keys = x @ self.W_key
        # input lai key weight sanga multiply garera keys banauxa

        queries = x @ self.W_query
        # input lai query weight sanga multiply garera queries banauxa

        values = x @ self.W_value
        # input lai value weight sanga multiply garera values banauxa

        attn_scores = queries @ keys.T  # omega
        # query ra key bich similarity nikalera attention score banauxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scale garera softmax lagako
        # dim=-1 le sabai tokens ma probability banauxa

        context_vec = attn_weights @ values
        # attention weight use garera final context vector banauxa

        return context_vec

In [36]:
torch.manual_seed(123)
# random values same auna ko lagi seed set gareko

sa_v1 = SelfAttention_v1(d_in, d_out)
# SelfAttention class ko object create gareko

print(sa_v1(inputs))
# inputs lai self-attention layer ma pass garera output print gareko

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [37]:
class SelfAttention_v2(nn.Module):
    # improved self-attention version

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        # linear layer use garera query banauxa (weight + optional bias)

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        # linear layer use garera key banauxa

        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # linear layer use garera value banauxa

    def forward(self, x):

        keys = self.W_key(x)
        # input lai key linear layer bata pass garera keys banauxa

        queries = self.W_query(x)
        # input lai query linear layer bata pass garera queries banauxa

        values = self.W_value(x)
        # input lai value linear layer bata pass garera values banauxa

        attn_scores = queries @ keys.T
        # query ra key ko dot product garera attention score nikalxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scale garera softmax lagauxa
        # dim=-1 le last dimension ma probability banauxa

        context_vec = attn_weights @ values
        # attention weights use garera final context vector banauxa

        return context_vec

In [38]:
torch.manual_seed(789)
# random initialization same auna ko lagi seed set gareko

sa_v2 = SelfAttention_v2(d_in, d_out)
# SelfAttention_v2 class ko object create gareko (linear layers use garera)

print(sa_v2(inputs))
# inputs lai self-attention v2 ma pass garera output print gareko

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


#causal attention

In [39]:
context_length = attn_scores.shape[0]
# attention score matrix ko size bata sequence length nikaleko

mask_simple = torch.tril(torch.ones(context_length, context_length))
# lower triangular matrix banako (future tokens mask garna)
# tril le diagonal samma ko lower part 1 banauxa, mathi ko 0 hunxa

print(mask_simple)
# mask matrix print gareko
# yesle future token attention block garna use hunxa

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [40]:
masked_simple = attn_weights * mask_simple
# attention weights lai mask sanga multiply gareko
# future tokens (upper triangle) lai 0 banauxa
# yesle model le future information herna sakdaina

print(masked_simple)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [41]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
# each row ko sum nikaleko
# keepdim=True le shape same rakxa (broadcasting ko lagi)

masked_simple_norm = masked_simple / row_sums
# mask gareko attention weights lai normalize gareko
# each row sum = 1 banauxa (probability jasto)

print(masked_simple_norm)
# final normalized masked attention weights print gareko

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [42]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
# upper triangular mask banako (future tokens ko part)
# diagonal=1 le diagonal mathi ko sabai elements 1 banauxa

masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
# mask bhayeko position haru ma -inf rakheko
# yesle softmax pachi tyo values 0 banauxa (completely ignore hunxa)

print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [43]:
attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=1)
# sqrt(d_k) le values stable banauxa

print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [44]:
torch.manual_seed(123)
# random behavior same auna ko lagi seed set gareko

dropout = torch.nn.Dropout(0.5)  # A
# dropout layer banako (0.5 probability le elements drop garxa)
# training ma overfitting reduce garna use hunxa

example = torch.ones(6, 6)  # B
# 6x6 matrix banako sabai values 1 banayera

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [45]:
torch.manual_seed(123)

print(dropout(attn_weights))
# overfitting reduce garna help garxa

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [46]:
batch = torch.stack((inputs, inputs), dim=0)
# same inputs lai batch ma combine gareko

print(batch.shape)
# yesle batch size, tokens, embedding dim dekhaunxa

torch.Size([2, 6, 3])


In [47]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.d_out = d_out

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)  # New
        # dropout layer banako (overfitting reduce garna)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        # upper triangular mask banako
        # future tokens lai block garna use hunxa
        # register_buffer le model sanga save hunxa tara train hudaina

    def forward(self, x):

        b, num_tokens, d_in = x.shape
        # batch size, tokens, embedding dimension nikaleko

        keys = self.W_key(x)
        # input lai key space ma transform gareko

        queries = self.W_query(x)
        # input lai query space ma transform gareko

        values = self.W_value(x)
        # input lai value space ma transform gareko

        attn_scores = queries @ keys.transpose(1, 2)
        # batch-wise attention score nikaleko
        # transpose(1,2) le token dimension swap garxa

        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        # future tokens ko attention -inf banako
        # softmax pachi tyo 0 hunxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scaled softmax lagako

        attn_weights = self.dropout(attn_weights)
        # dropout apply gareko (randomly kehi attention drop hunxa)

        context_vec = attn_weights @ values
        # final context vector banako (weighted sum)

        return context_vec

In [48]:
torch.manual_seed(123)

context_length = batch.shape[1]
# sequence length (token count per sample) nikaleko

ca = CausalAttention(d_in, d_out, context_length, 0.0)
# causal attention model create gareko
# 0.0 dropout means training ma pani dropout disable jasto hunxa

context_vecs = ca(batch)
# batch input lai causal attention layer ma pass gareko
# context-aware representation banauxa

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])
